In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("../utils"))

from pathlib import Path
import re
import torch as th
import numpy as np
import gymnasium as gym
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.policies import ActorCriticPolicy
import torch.nn as nn
from imitation.util import logger as imit_logger
from torch.utils.data import DataLoader
import datetime
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, VecFrameStack
from mario import make_env
from sbx import PPO

import gc
from IPython import get_ipython
import cv2

from utils import get_last_index

current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = f"./tensorboard_ppo/run_{current_time}"
custom_logger = imit_logger.configure(log_path, ["tensorboard", "stdout"])

gc.collect()
th.cuda.empty_cache()

train_path = 'ppo/'
sub_train_path = train_path + "steps/"

os.makedirs(train_path, exist_ok=True)
os.makedirs(sub_train_path, exist_ok=True)


ENT_WEIGHT= 1e-3
BATCH_SIZE= 128
LEARNING_RATE= 1e-4
NUMBER_TRAIN = 200
N_ENVS=4
N_STEPS= 1024 # 2048


env = SubprocVecEnv(
    [make_env(i, human=False) for i in range(N_ENVS)]
)
env = VecFrameStack(env, 4, channels_order='last')

last_index_imitation = get_last_index(str(sub_train_path), "ppo_policy", "zip")

print(f"Last index ppo: {last_index_imitation}")

model = None

policy_kwargs = dict(
    net_arch=dict(pi=[256, 256], vf=[256, 256]),
)

if last_index_imitation < 0:
    model = PPO(
        policy="CnnPolicy", 
        env=env,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        ent_coef=ENT_WEIGHT,
        n_steps=N_STEPS * N_ENVS,
        verbose=1,
        policy_kwargs=policy_kwargs,
        device="cpu"
    )
    
    print("Starting from zero model")
else:
    model = PPO.load(
        f"./ppo/steps/ppo_policy{last_index_imitation}.zip", 
        env=env,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        ent_coef=ENT_WEIGHT,
        n_steps=N_STEPS * N_ENVS,
        verbose=1,
        policy_kwargs=policy_kwargs,
        device="cpu"
    )
    print(f"Loading model from ppo number: {last_index_imitation}")

model.set_logger(custom_logger)

current_epoch = last_index_imitation + 1

run_name = f"PPO_FineTuning_{current_time}"


for i in range(NUMBER_TRAIN):
    model.learn(
        total_timesteps=N_STEPS * N_ENVS * 2,
        reset_num_timesteps=False,
        tb_log_name=run_name
    )
    model.save(f"./ppo/steps/ppo_policy{current_epoch}.zip")

    current_epoch += 1
    
gc.collect()
del model

th.cuda.empty_cache()

print("Force cell kernel reset")
get_ipython().kernel.do_shutdown(restart=True)

D:\anaconda3\envs\env311\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Last index ppo: 8
Wrapping the env in a VecTransposeImage.
Loading model from ppo number: 8
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 259      |
|    ep_rew_mean     | -6.23    |
| time/              |          |
|    fps             | 40       |
|    iterations      | 1        |
|    time_elapsed    | 400      |
|    total_timesteps | 163840   |
---------------------------------
--------------------------------------
| rollout/                |          |
|    ep_len_mean          | 267      |
|    ep_rew_mean          | -6.25    |
| time/                   |          |
|    fps                  | 38       |
|    iterations           | 1        |
|    time_elapsed         | 420      |
|    total_timesteps      | 180224   |
| train/                  |          |
|    approx_kl            | 0.0177   |
|    clip_fraction        | 0.261    |
|    clip_range           | 0.2      |
|    entropy_loss         | -1.47    |
|    explained_varianc